# ConvMAE Pre-Training — Parquet Inputs

Trains ConvMAE on radio-map data stored as Parquet files.  
Each file contains one sample: a flat `map` column of 810,000 float64 dBm values (900×900).  
The raw scalar map is normalised and rendered via jet colourmap → RGB image for ConvMAE.

**All compatibility patches pre-applied:**
- `timm==0.3.2` + `torch._six` monkey-patch for PyTorch 2.x
- `np.float` → `np.float32` (NumPy 1.24+)
- `misc.add_weight_decay` → `optim_factory.add_weight_decay`
- `torch.cuda.amp.GradScaler` / `autocast` → `torch.amp` equivalents

**Dataset:** `labels_parquet_v2/{training,validation,testing}`

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. GPU Check

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')

## 3. Install Dependencies

In [ ]:
# timm 0.3.2 required by ConvMAE; monkey-patch torch._six for PyTorch 2.x
!pip uninstall timm -y -q
!pip install timm==0.3.2 einops pyarrow pandas -q

import sys, collections.abc, torch
if not hasattr(torch, '_six'):
    class _Six:
        container_abcs = collections.abc
    sys.modules['torch._six'] = _Six()

import timm
print('timm version:', timm.__version__)

## 4. Clone ConvMAE & Apply Compatibility Patches

In [ ]:
import os

REPO_DIR = '/content/ConvMAE'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Alpha-VL/ConvMAE.git {REPO_DIR}
else:
    print('Already cloned — pulling.')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)

# Patch 1: np.float removed in NumPy 1.24
!sed -i 's/np\.float\b/np.float32/g' /content/ConvMAE/util/pos_embed.py

# Patch 2: deprecated GradScaler API
!sed -i 's/torch.cuda.amp.GradScaler()/torch.amp.GradScaler("cuda")/g' /content/ConvMAE/util/misc.py

# Patch 3: deprecated autocast API
!sed -i "s/torch.cuda.amp.autocast()/torch.amp.autocast('cuda')/g" /content/ConvMAE/engine_pretrain.py

print('All patches applied.')
!ls

## 5. ⚙️ Dataset Configuration

**Edit `TOTAL_SAMPLES` to change how many Parquet files are used.**  
Examples: `1000`, `5000`, `10000`, or `None` for the full dataset.

In [ ]:
# ════════════════════════════════════════════════════
#  CHANGE THESE VALUES TO ADJUST DATASET SIZE / SPLIT
# ════════════════════════════════════════════════════
TOTAL_SAMPLES = 1000   # e.g. 1000 | 5000 | 10000 | None (full)
SPLIT_TRAIN   = 0.80
SPLIT_VAL     = 0.10
SPLIT_TEST    = 0.10
RANDOM_SEED   = 42

# dBm range for normalisation
DBM_MIN = -140.0
DBM_MAX = -40.0
# ════════════════════════════════════════════════════

PQ_BASE      = '/content/drive/MyDrive/Senior Design/dataset/labels_parquet_v2'
DRIVE_TRAIN  = f'{PQ_BASE}/training'
DRIVE_VAL    = f'{PQ_BASE}/validation'
DRIVE_TEST   = f'{PQ_BASE}/testing'

assert abs(SPLIT_TRAIN + SPLIT_VAL + SPLIT_TEST - 1.0) < 1e-6
print(f'Target samples : {TOTAL_SAMPLES if TOTAL_SAMPLES else "ALL"}')
print(f'Split          : {SPLIT_TRAIN:.0%} train / {SPLIT_VAL:.0%} val / {SPLIT_TEST:.0%} test')

## 6. Build Sampled File Lists (80/10/10)

In [ ]:
import glob, random
import numpy as np

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

all_files = []
for src in [DRIVE_TRAIN, DRIVE_VAL, DRIVE_TEST]:
    all_files.extend(sorted(glob.glob(os.path.join(src, '*.parquet'))))

if not all_files:
    raise FileNotFoundError(f'No .parquet files found under {PQ_BASE}')

random.shuffle(all_files)
if TOTAL_SAMPLES:
    all_files = all_files[:TOTAL_SAMPLES]

N       = len(all_files)
n_train = int(N * SPLIT_TRAIN)
n_val   = int(N * SPLIT_VAL)
n_test  = N - n_train - n_val

TRAIN_FILES = all_files[:n_train]
VAL_FILES   = all_files[n_train : n_train + n_val]
TEST_FILES  = all_files[n_train + n_val :]

print(f'Total : {N}  |  Train : {n_train}  |  Val : {n_val}  |  Test : {n_test}')

## 7. Custom Parquet Dataset + Sanity Check

Reads each `.parquet` file, reshapes the flat `map` column → 900×900,  
normalises dBm to [0,1], applies jet colourmap → RGB PIL Image.

In [ ]:
dataset_code = '''
"""
parquet_dataset.py — PyTorch Dataset for radio-map Parquet files.
Each .parquet file contains one row with a flat `map` column.
"""
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset


def jet_colormap(v01):
    """Apply matplotlib jet colourmap; returns uint8 (H,W,3)."""
    try:
        import matplotlib.cm as cm
        return (cm.jet(v01)[..., :3] * 255).astype(np.uint8)
    except Exception:
        v = np.clip(v01, 0, 1)
        r = np.clip(1.5 - abs(4*v - 3), 0, 1)
        g = np.clip(1.5 - abs(4*v - 2), 0, 1)
        b = np.clip(1.5 - abs(4*v - 1), 0, 1)
        return (np.stack([r, g, b], -1) * 255).astype(np.uint8)


class ParquetRadioMapDataset(Dataset):
    def __init__(self, file_list, transform=None, dbm_min=-140.0, dbm_max=-40.0):
        self.files     = file_list
        self.transform = transform
        self.dbm_min   = dbm_min
        self.dbm_max   = dbm_max

    def __len__(self):
        return len(self.files)

    def _load(self, path):
        df  = pd.read_parquet(path, columns=["map", "map_width", "map_height"])
        row = df.iloc[0]
        w, h = int(row["map_width"]), int(row["map_height"])
        grid = np.array(row["map"], dtype=np.float32).reshape(h, w)
        norm = np.clip((grid - self.dbm_min) / (self.dbm_max - self.dbm_min), 0.0, 1.0)
        return Image.fromarray(jet_colormap(norm), mode="RGB")

    def __getitem__(self, idx):
        img = self._load(self.files[idx])
        if self.transform:
            img = self.transform(img)
        return img, 0   # label unused during pre-training
'''

with open('/content/ConvMAE/parquet_dataset.py', 'w') as f:
    f.write(dataset_code)
print('parquet_dataset.py written.')

In [ ]:
# Sanity-check: load and display one converted sample
import sys, matplotlib.pyplot as plt
sys.path.insert(0, '/content/ConvMAE')
from parquet_dataset import ParquetRadioMapDataset

ds_check = ParquetRadioMapDataset(TRAIN_FILES[:5])
img, _ = ds_check[0]
print(f'Sample size: {img.size}  mode: {img.mode}')

plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.title('Parquet → jet RGB (sample 0)')
plt.axis('off')
plt.tight_layout()
plt.show()

## 8. Write Parquet Pre-Train Launcher

In [ ]:
launcher_code = r'''
"""
run_pretrain_parquet.py — ConvMAE pre-training with Parquet radio-map files.
Single-GPU Colab-compatible. Mirrors main_pretrain.py but uses ParquetRadioMapDataset.
"""
import argparse, datetime, json, os, sys, time
from pathlib import Path
import numpy as np
import torch
import torch.backends.cudnn as cudnn
from torch.utils.tensorboard import SummaryWriter
import torchvision.transforms as transforms

sys.path.insert(0, '/content/ConvMAE')
import models_convmae
import util.misc as misc
import timm.optim.optim_factory as optim_factory
from engine_pretrain import train_one_epoch
from parquet_dataset import ParquetRadioMapDataset


def get_args_parser():
    p = argparse.ArgumentParser('ConvMAE Parquet pre-training', add_help=False)
    p.add_argument('--train_files',   nargs='+', required=True)
    p.add_argument('--dbm_min',       default=-140.0, type=float)
    p.add_argument('--dbm_max',       default=-40.0,  type=float)
    p.add_argument('--model',         default='convmae_convvit_base_patch16', type=str)
    p.add_argument('--input_size',    default=224, type=int)
    p.add_argument('--mask_ratio',    default=0.75, type=float)
    p.add_argument('--batch_size',    default=32, type=int)
    p.add_argument('--epochs',        default=50, type=int)
    p.add_argument('--warmup_epochs', default=5,  type=int)
    p.add_argument('--blr',           default=1.5e-4, type=float)
    p.add_argument('--weight_decay',  default=0.05,   type=float)
    p.add_argument('--min_lr',        default=0.0,    type=float)
    p.add_argument('--accum_iter',    default=1,      type=int)
    p.add_argument('--norm_pix_loss', action='store_true')
    p.add_argument('--num_workers',   default=2,   type=int)
    p.add_argument('--pin_mem',       action='store_true')
    p.add_argument('--no_pin_mem',    action='store_false', dest='pin_mem')
    p.set_defaults(pin_mem=True)
    p.add_argument('--device',        default='cuda', type=str)
    p.add_argument('--seed',          default=0,   type=int)
    p.add_argument('--output_dir',    default='',  type=str)
    p.add_argument('--log_dir',       default='',  type=str)
    p.add_argument('--save_ckpt_freq',default=10,  type=int)
    p.add_argument('--resume',        default='',  type=str)
    p.add_argument('--start_epoch',   default=0,   type=int)
    p.add_argument('--world_size',    default=1,   type=int)
    p.add_argument('--local_rank',    default=-1,  type=int)
    p.add_argument('--dist_on_itp',   action='store_true')
    p.add_argument('--dist_url',      default='env://', type=str)
    return p


def main(args):
    misc.init_distributed_mode(args)
    print('Config:', json.dumps(vars(args), indent=2))

    device = torch.device(args.device)
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    cudnn.benchmark = True

    transform_train = transforms.Compose([
        transforms.RandomResizedCrop(args.input_size, scale=(0.2, 1.0), interpolation=3),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    dataset_train = ParquetRadioMapDataset(
        file_list=args.train_files,
        transform=transform_train,
        dbm_min=args.dbm_min,
        dbm_max=args.dbm_max,
    )
    print(f'Train dataset: {len(dataset_train)} samples')

    sampler = torch.utils.data.RandomSampler(dataset_train)
    log_writer = SummaryWriter(log_dir=args.log_dir) if args.log_dir else None

    loader = torch.utils.data.DataLoader(
        dataset_train, sampler=sampler,
        batch_size=args.batch_size,
        num_workers=args.num_workers,
        pin_memory=args.pin_mem,
        drop_last=True,
    )

    model = models_convmae.__dict__[args.model](norm_pix_loss=args.norm_pix_loss)
    model.to(device)
    model_without_ddp = model
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Trainable params: {n/1e6:.1f}M')

    eff_batch = args.batch_size * args.accum_iter
    args.lr   = args.blr * eff_batch / 256
    print(f'Effective batch: {eff_batch} | LR: {args.lr:.6f}')

    param_groups = optim_factory.add_weight_decay(model, args.weight_decay)
    optimizer    = torch.optim.AdamW(param_groups, lr=args.lr, betas=(0.9, 0.95))
    loss_scaler  = misc.NativeScalerWithGradNormCount()

    if args.resume:
        misc.load_model(args=args, model_without_ddp=model_without_ddp,
                        optimizer=optimizer, loss_scaler=loss_scaler)

    print(f'\nStart training for {args.epochs} epochs')
    start = time.time()
    for epoch in range(args.start_epoch, args.epochs):
        stats = train_one_epoch(
            model, loader, optimizer, device, epoch,
            loss_scaler, log_writer=log_writer, args=args,
        )
        if args.output_dir and (
            (epoch + 1) % args.save_ckpt_freq == 0 or epoch + 1 == args.epochs
        ):
            misc.save_model(
                args=args, model=model,
                model_without_ddp=model_without_ddp,
                optimizer=optimizer, loss_scaler=loss_scaler, epoch=epoch,
            )
        if args.output_dir:
            log = {**{f'train_{k}': v for k, v in stats.items()}, 'epoch': epoch}
            with open(os.path.join(args.output_dir, 'log.txt'), 'a') as f:
                f.write(json.dumps(log) + '\n')

    print(f'Done in {datetime.timedelta(seconds=int(time.time()-start))}')


if __name__ == '__main__':
    os.environ.setdefault('RANK', '0')
    os.environ.setdefault('LOCAL_RANK', '0')
    os.environ.setdefault('WORLD_SIZE', '1')
    os.environ.setdefault('MASTER_ADDR', 'localhost')
    os.environ.setdefault('MASTER_PORT', '12345')

    parser = argparse.ArgumentParser(parents=[get_args_parser()])
    args   = parser.parse_args()
    if args.output_dir:
        Path(args.output_dir).mkdir(parents=True, exist_ok=True)
    main(args)
'''

with open('/content/ConvMAE/run_pretrain_parquet.py', 'w') as f:
    f.write(launcher_code)
print('run_pretrain_parquet.py written.')

## 9. ⚙️ Training Hyperparameters

In [ ]:
import os

MODEL          = 'convmae_convvit_base_patch16'
INPUT_SIZE     = 224
BATCH_SIZE     = 32       # Reduce to 16 if OOM
EPOCHS         = 50       # Rapid ablation default
WARMUP_EPOCHS  = 5
LR             = 1.5e-4
MASK_RATIO     = 0.75
NUM_WORKERS    = 2
SAVE_CKPT_FREQ = 10

OUTPUT_DIR = '/content/drive/MyDrive/Senior Design/checkpoints/convmae_parquet'
LOG_DIR    = OUTPUT_DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Output dir:', OUTPUT_DIR)

## 10. Run Pre-Training

In [ ]:
import subprocess, sys, os

os.environ['MASTER_ADDR'] = 'localhost'
os.environ['MASTER_PORT'] = '12345'

cmd = [
    sys.executable, '/content/ConvMAE/run_pretrain_parquet.py',
    '--train_files',    *TRAIN_FILES,    # pass sampled file list directly
    '--dbm_min',        str(DBM_MIN),
    '--dbm_max',        str(DBM_MAX),
    '--model',          MODEL,
    '--input_size',     str(INPUT_SIZE),
    '--mask_ratio',     str(MASK_RATIO),
    '--batch_size',     str(BATCH_SIZE),
    '--epochs',         str(EPOCHS),
    '--warmup_epochs',  str(WARMUP_EPOCHS),
    '--blr',            str(LR),
    '--num_workers',    str(NUM_WORKERS),
    '--pin_mem',
    '--output_dir',     OUTPUT_DIR,
    '--log_dir',        LOG_DIR,
    '--save_ckpt_freq', str(SAVE_CKPT_FREQ),
    '--dist_url',       'env://',
]

print(f'Training on {len(TRAIN_FILES)} Parquet files for {EPOCHS} epochs.')
print('='*60)

LOG_LINES_PQ = []
process = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='')
    LOG_LINES_PQ.append(line)

process.wait()
if process.returncode != 0:
    raise RuntimeError(f'Training failed (code {process.returncode})')
print('\nTraining complete!')

## 11. Parse Training Log

In [ ]:
import json, re
import numpy as np

epochs_pq, losses_pq, lrs_pq = [], [], []

log_path = os.path.join(OUTPUT_DIR, 'log.txt')
if os.path.exists(log_path):
    with open(log_path) as f:
        for line in f:
            try:
                d = json.loads(line.strip())
                epochs_pq.append(int(d['epoch']))
                losses_pq.append(float(d.get('train_loss', float('nan'))))
                lrs_pq.append(float(d.get('train_lr', float('nan'))))
            except Exception:
                pass

if not epochs_pq:
    for line in LOG_LINES_PQ:
        m = re.search(r'\[?(\d+)/(\d+)\]?.*?loss[:\s]+([0-9.]+)', line, re.IGNORECASE)
        if m:
            epochs_pq.append(int(m.group(1)))
            losses_pq.append(float(m.group(3)))
            lrs_pq.append(float('nan'))

print(f'Parsed {len(epochs_pq)} epoch records.')
if losses_pq:
    valid = [l for l in losses_pq if not np.isnan(l)]
    print(f'Loss — min: {min(valid):.5f}  max: {max(valid):.5f}  final: {valid[-1]:.5f}')

## 12. 📊 Performance Visualisation

Six-panel dashboard: learning curve, RMSE, LR schedule, loss histogram, epoch delta, log-scale convergence.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import seaborn as sns
from matplotlib.ticker import MaxNLocator

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

if not losses_pq:
    print('No training data to plot. Run training first.')
else:
    ep   = np.array(epochs_pq, dtype=float)
    mse  = np.array(losses_pq, dtype=float)
    lrs  = np.array(lrs_pq,    dtype=float)
    rmse = np.sqrt(np.clip(mse, 0, None))

    def ema(v, a=0.3):
        out = np.empty_like(v); out[0] = v[0]
        for i in range(1, len(v)): out[i] = a*v[i] + (1-a)*out[i-1]
        return out

    fig = plt.figure(figsize=(18, 12))
    fig.suptitle(
        f'ConvMAE Parquet Pre-Training  |  {len(ep)} epochs  |  {n_train} train samples',
        fontsize=15, fontweight='bold', y=1.01
    )
    gs = gridspec.GridSpec(2, 3, hspace=0.42, wspace=0.35)

    # Panel 1: Learning Curve (MSE)
    ax = fig.add_subplot(gs[0, 0])
    ax.plot(ep, mse, color='steelblue', alpha=0.35, lw=1, label='MSE raw')
    ax.plot(ep, ema(mse), color='steelblue', lw=2.5, label='MSE smoothed')
    ax.set_title('Learning Curve (MSE)'); ax.set_xlabel('Epoch'); ax.set_ylabel('MSE')
    ax.legend(); ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    # Panel 2: RMSE
    ax = fig.add_subplot(gs[0, 1])
    ax.plot(ep, rmse, color='tomato', lw=2.5)
    ax.fill_between(ep, rmse, alpha=0.15, color='tomato')
    ax.set_title('RMSE'); ax.set_xlabel('Epoch'); ax.set_ylabel('RMSE')
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    # Panel 3: LR schedule
    ax = fig.add_subplot(gs[0, 2])
    if not np.all(np.isnan(lrs)):
        ax.plot(ep, lrs, color='darkorange', lw=2.5)
        ax.set_ylabel('LR')
    else:
        ax.text(0.5, 0.5, 'LR not\nlogged', ha='center', va='center',
                transform=ax.transAxes, color='grey', fontsize=12)
    ax.set_title('LR Schedule'); ax.set_xlabel('Epoch')
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    # Panel 4: MSE histogram
    ax = fig.add_subplot(gs[1, 0])
    sns.histplot(mse[~np.isnan(mse)], kde=True, ax=ax, color='mediumseagreen')
    ax.set_title('MSE Distribution'); ax.set_xlabel('MSE'); ax.set_ylabel('Count')

    # Panel 5: Epoch-over-epoch delta
    ax = fig.add_subplot(gs[1, 1])
    if len(mse) > 1:
        d = np.diff(mse)
        ax.bar(ep[1:], d,
               color=['tomato' if x > 0 else 'steelblue' for x in d],
               edgecolor='none', alpha=0.8)
        ax.axhline(0, color='black', lw=0.8)
    ax.set_title('Epoch-over-Epoch Δ MSE'); ax.set_xlabel('Epoch'); ax.set_ylabel('Δ MSE')
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    # Panel 6: Log-scale convergence
    ax = fig.add_subplot(gs[1, 2])
    mask = mse > 0
    if mask.any():
        ax.semilogy(ep[mask], mse[mask], color='purple', lw=2.5)
    ax.set_title('Convergence (log scale)'); ax.set_xlabel('Epoch'); ax.set_ylabel('log(MSE)')
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    plt.tight_layout()
    SAVE_FIG = os.path.join(OUTPUT_DIR, 'training_metrics_parquet.png')
    plt.savefig(SAVE_FIG, dpi=150, bbox_inches='tight')
    plt.show()
    print('Figure saved:', SAVE_FIG)

## 13. Summary Statistics Table

In [ ]:
import pandas as pd

if losses_pq:
    df = pd.DataFrame({
        'Epoch': ep.astype(int),
        'MSE'  : mse,
        'RMSE' : rmse,
        'LR'   : lrs,
    })
    print(df.describe().round(6).to_string())
    best = df.loc[df.MSE.idxmin()]
    print(f'\nBest MSE  : {best.MSE:.6f}  @ epoch {int(best.Epoch)}')
    print(f'Final MSE : {df.MSE.iloc[-1]:.6f}')
    pct = (df.MSE.iloc[0] - df.MSE.iloc[-1]) / df.MSE.iloc[0] * 100
    print(f'Total reduction: {pct:.1f}%')

    csv_path = os.path.join(OUTPUT_DIR, 'training_log_parquet.csv')
    df.to_csv(csv_path, index=False)
    print('CSV saved:', csv_path)

## 14. Verify Checkpoints

In [ ]:
import glob
ckpts = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.pth')))
print(f'Checkpoints ({len(ckpts)}):')
for c in ckpts:
    print(f'  {os.path.basename(c)}  ({os.path.getsize(c)/1e6:.1f} MB)')